#**Laboratorio Práctico 06: Python Analytics Nivel Experto (Vectorización y Agrupación)**

# Caso de Negocio: Algoritmo de Bonificación y Reportes Automatizados

Usted es el Data Engineer Senior de TechMarket, un retail multinacional. El equipo de Recursos
Humanos le ha enviado la base de datos del desempeño trimestral de su fuerza de ventas.

Su misión tiene tres etapas:
1. Crear un algoritmo vectorizado que clasifique el rendimiento y calcule bonos
automáticamente.
2. Generar un resumen gerencial (agrupado) para el Directorio.
3. Automatizar la creación de reportes individuales por región.

Preparación del Entorno (Ejecute esta celda primero)

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Generación de la data de desempeño
data_desempeno = {
'id_empleado': ['E01', 'E02', 'E03', 'E04', 'E05', 'E06',
'E07', 'E08'],
'region': ['Norte', 'Norte', 'Sur', 'Sur', 'Centro', 'Norte',
'Sur', 'Centro'],
'ventas_usd': [12000, 4500, 8000, 15000, 3000, np.nan, 9500,
11000],
'satisfaccion_cliente': [4.8, 3.2, 4.1, 4.9, 2.5, 4.0, 3.8,
4.7],
'meses_antiguedad': [24, 6, 12, 36, 4, 18, 9, 48]
}
df_ventas = pd.DataFrame(data_desempeno)
display(df_ventas.head())

,id_empleado,region,ventas_usd,satisfaccion_cliente,meses_antiguedad
0,E01,Norte,12000.0,4.8,24
1,E02,Norte,4500.0,3.2,6
2,E03,Sur,8000.0,4.1,12
3,E04,Sur,15000.0,4.9,36
4,E05,Centro,3000.0,2.5,4


**Parte 1: El Motor de Reglas (np.where Avanzado)**


El directorio ha establecido reglas estrictas para evaluar a los empleados y asignarles su bono
trimestral. Debe crear dos nuevas columnas en df_ventas aplicando lógica condicional.



**1. Columna categoria_rendimiento: Utilice np.where anidado para clasificar a cada empleado:**

○ Si sus ventas_usd son mayores a 10,000 Y su satisfaccion_cliente es
mayor o igual a 4.5 -> "Top Talent".

○ Si sus ventas_usd son mayores o iguales a 5,000 -> "Regular".

○ Si no cumple ninguna de las anteriores (o si las ventas son nulas) -> "Bajo
Rendimiento".

In [ ]:
df_ventas['categoria_rendimiento'] = np.where(df_ventas['ventas_usd'] > 10000, 'Top Talent',
                                          np.where(df_ventas['ventas_usd']>5000,'Regular','Bajo Rendimiento'))

In [ ]:
display(df_ventas)

,id_empleado,region,ventas_usd,satisfaccion_cliente,meses_antiguedad,categoria_rendimiento
0,E01,Norte,12000.0,4.8,24,Top Talent
1,E02,Norte,4500.0,3.2,6,Bajo Rendimiento
2,E03,Sur,8000.0,4.1,12,Regular
3,E04,Sur,15000.0,4.9,36,Top Talent
4,E05,Centro,3000.0,2.5,4,Bajo Rendimiento
5,E06,Norte,NaN,4.0,18,Bajo Rendimiento
6,E07,Sur,9500.0,3.8,9,Regular
7,E08,Centro,11000.0,4.7,48,Top Talent


**2. Columna bono_usd: Utilice np.where para calcular matemáticamente el bono**

○ Si es "Top Talent", el bono es el 15% de sus ventas_usd.

○ Si es "Regular", el bono es el 5% de sus ventas_usd.

○ Si es "Bajo Rendimiento", el bono es 0.

In [ ]:
df_ventas['bono_usd'] = np.where(
    df_ventas['categoria_rendimiento']=='Top Talent', df_ventas['ventas_usd']*0.15,
    np.where(df_ventas['categoria_rendimiento']=='Regular', df_ventas['ventas_usd']*0.05,0)
)

In [ ]:
display(df_ventas)

,id_empleado,region,ventas_usd,satisfaccion_cliente,meses_antiguedad,categoria_rendimiento,bono_usd
0,E01,Norte,12000.0,4.8,24,Top Talent,1800.0
1,E02,Norte,4500.0,3.2,6,Bajo Rendimiento,0.0
2,E03,Sur,8000.0,4.1,12,Regular,400.0
3,E04,Sur,15000.0,4.9,36,Top Talent,2250.0
4,E05,Centro,3000.0,2.5,4,Bajo Rendimiento,0.0
5,E06,Norte,NaN,4.0,18,Bajo Rendimiento,0.0
6,E07,Sur,9500.0,3.8,9,Regular,475.0
7,E08,Centro,11000.0,4.7,48,Top Talent,1650.0


**Parte 2: El Resumen Gerencial (groupby con agregación múltiple)**


El CEO no quiere ver fila por fila, quiere un resumen estratégico por región.

1. Utilice el método .groupby() para agrupar los datos por region y
categoria_rendimiento.
2. Asegúrese de utilizar el parámetro as_index=False para mantener un formato
tabular limpio.
3. Utilice el método .agg() para calcular exactamente lo siguiente en el mismo paso:

○ La suma total de ventas_usd por grupo.

○ La suma total de los bono_usd que se pagarán.

○ El promedio (mean) de la satisfaccion_cliente.

○ El conteo (count) de empleados (id_empleado) en ese grupo.

4. Guarde este resumen en la variable df_resumen_gerencial y muéstrelo en pantala.

In [ ]:
df_ventas_region_rendimiento = df_ventas.groupby(
    (['region','categoria_rendimiento']), as_index = False).agg({'ventas_usd':'sum',
                                                                 'bono_usd':'sum',
                                                                 'satisfaccion_cliente':'mean',
                                                                 'id_empleado':'count'})

In [ ]:
display(df_ventas_region_rendimiento)

,region,categoria_rendimiento,ventas_usd,bono_usd,satisfaccion_cliente,id_empleado
0,Centro,Bajo Rendimiento,3000.0,0.0,2.50,1
1,Centro,Top Talent,11000.0,1650.0,4.70,1
2,Norte,Bajo Rendimiento,4500.0,0.0,3.60,2
3,Norte,Top Talent,12000.0,1800.0,4.80,1
4,Sur,Regular,17500.0,875.0,3.95,2
5,Sur,Top Talent,15000.0,2250.0,4.90,1


**Parte 3: Automatización de Exportación (enumerate + groupby)**

Usted debe enviarle a cada Gerente Regional un reporte solo con los datos de sus empleados.
En lugar de filtrar la tabla región por región manualmente, automatizaremos el proceso.
Misiones:

1. Genere un objeto groupby agrupando df_ventas únicamente por la columna
region.

2. Utilice un bucle for combinando enumerate (iniciando en 1) para iterar sobre este
objeto agrupado.

3. En cada iteración del bucle, debe imprimir en consola un texto dinámico exactamente como este:

* "Generando Reporte 1 - Región: Norte"




* "Generando Reporte 2 - Región: Sur" (etc.)

4. Debajo de cada título, imprima el "mini DataFrame" que le entrega el bucle, pero únicamente mostrando las columnas id_empleado, categoria_rendimiento y bono_usd.

In [ ]:
#Generando objeto de agrupación por región

df_ventas_region = df_ventas.groupby('region')

In [ ]:
#Generando Bucle "for" iterando el objeto agrupado
for region_agrupacion in enumerate(df_ventas_region):
  print(region_agrupacion)

(0, ('Centro',   id_empleado  region  ventas_usd  satisfaccion_cliente  meses_antiguedad  \
4         E05  Centro      3000.0                   2.5                 4   
7         E08  Centro     11000.0                   4.7                48   

  categoria_rendimiento  bono_usd  
4      Bajo Rendimiento       0.0  
7            Top Talent    1650.0  ))
(1, ('Norte',   id_empleado region  ventas_usd  satisfaccion_cliente  meses_antiguedad  \
0         E01  Norte     12000.0                   4.8                24   
1         E02  Norte      4500.0                   3.2                 6   
5         E06  Norte         NaN                   4.0                18   

  categoria_rendimiento  bono_usd  
0            Top Talent    1800.0  
1      Bajo Rendimiento       0.0  
5      Bajo Rendimiento       0.0  ))
(2, ('Sur',   id_empleado region  ventas_usd  satisfaccion_cliente  meses_antiguedad  \
2         E03    Sur      8000.0                   4.1                12   
3         E04 

In [ ]:
#Modificando el bucle incluyendo el sub dataframe
for indice, (nombre_region, sub_df) in enumerate(df_ventas_region,start=1):
  print(f'=== Generando Reporte {indice} - Región: {nombre_region}====\n')
  print(f'{sub_df[['id_empleado','categoria_rendimiento','bono_usd']]}\n')


=== Generando Reporte 1 - Región: Centro====

  id_empleado categoria_rendimiento  bono_usd
4         E05      Bajo Rendimiento       0.0
7         E08            Top Talent    1650.0

=== Generando Reporte 2 - Región: Norte====

  id_empleado categoria_rendimiento  bono_usd
0         E01            Top Talent    1800.0
1         E02      Bajo Rendimiento       0.0
5         E06      Bajo Rendimiento       0.0

=== Generando Reporte 3 - Región: Sur====

  id_empleado categoria_rendimiento  bono_usd
2         E03               Regular     400.0
3         E04            Top Talent    2250.0
6         E07               Regular     475.0

